# Grid Network Analysis

Mainline analysis of the grid topology together with the processed prosumer dataset.


In [ ]:
from pathlib import Path
import sys
import warnings

project_root = Path.cwd().resolve()
while project_root != project_root.parent and not (project_root / "configs").exists():
    project_root = project_root.parent
if not (project_root / "configs").exists():
    raise RuntimeError("Could not locate the project root from the notebook working directory.")
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

import matplotlib.pyplot as plt
import networkx as nx
import numpy as np
import pandas as pd
import pandapower as pp
from IPython.display import display

from configs import compose_experiment_config
from data.loaders.prosumer import ProsumerDataset
from envs.grid.core.net_builder import apply_bus_injections, build_simbench_net

warnings.filterwarnings("ignore")
plt.style.use("seaborn-v0_8-whitegrid")
pd.set_option("display.max_columns", 100)
pd.set_option("display.max_rows", 200)


In [ ]:
cfg = compose_experiment_config(
    profile="debug",
    algorithm="MADDPG",
    forecast_type="perfect",
    data_dir=project_root / "data",
    runtime_mode="performance",
    seed=0,
    require_cuda=False,
)

agent_bus_ids = list(map(int, cfg.grid.agent_bus_ids))
dataset = ProsumerDataset(
    data_dir=cfg.data.data_dir,
    episode_length=1,
    n_agents=len(agent_bus_ids),
    agent_profiles=list(cfg.data.agent_profiles),
    year=cfg.data.test_year,
    load_components=list(cfg.data.load_components),
    pv_reference=cfg.data.pv_reference,
    pv_capacity_kw=list(cfg.data.pv_capacity_kw),
    load_scale=list(cfg.data.load_scale),
    pv_scale=list(cfg.data.pv_scale),
    storage_scale=list(cfg.data.storage_scale),
    node_ids=agent_bus_ids,
)
first_episode = dataset.get_episode(0)
metadata = first_episode["meta"]

timestamps = pd.to_datetime(dataset._timestamps)
load_values = np.asarray(dataset._signals["load"], dtype=float)
pv_values = np.asarray(dataset._signals["pv"], dtype=float)
price_values = np.asarray(dataset._signals["price"], dtype=float)

project_year_df = pd.DataFrame({
    "timestamp": timestamps,
    "price": price_values,
    **{f"load{i + 1}": load_values[:, i] for i in range(load_values.shape[1])},
    **{f"pv{i + 1}": pv_values[:, i] for i in range(pv_values.shape[1])},
})
project_year_df["agent_total_load_kw"] = load_values.sum(axis=1)
project_year_df["agent_total_pv_kw"] = pv_values.sum(axis=1)
project_year_df["agent_total_net_kw"] = project_year_df["agent_total_load_kw"] - project_year_df["agent_total_pv_kw"]

raw_net = build_simbench_net(cfg.grid.sb_code)
runtime_summary = pd.Series(
    {
        "project_root": str(project_root),
        "grid_code": cfg.grid.sb_code,
        "agent_bus_ids": agent_bus_ids,
        "bus_count": int(len(raw_net.bus)),
        "line_count": int(len(raw_net.line)),
        "trafo_count": int(len(raw_net.trafo)),
        "dataset_rows": int(len(project_year_df)),
        "time_start": str(project_year_df["timestamp"].min()),
        "time_end": str(project_year_df["timestamp"].max()),
        "dt_hours": float(cfg.env.dt),
        "battery_capacity_kwh_each": float(cfg.env.battery_capacity),
        "battery_max_charge_discharge_kw_each": float(cfg.env.max_charge_rate),
    }
)
display(runtime_summary)


In [ ]:
def runpp_with_fallback(net, pf_solver="nr"):
    attempts = [
        {"algorithm": pf_solver, "numba": False, "calculate_voltage_angles": False, "init": "flat"},
        {"algorithm": "bfsw", "numba": False, "calculate_voltage_angles": False, "init": "flat"},
        {"algorithm": pf_solver, "numba": False, "calculate_voltage_angles": False, "init": "auto"},
    ]
    errors = []
    for attempt in attempts:
        try:
            pp.runpp(net, verbose=False, **attempt)
            return True, ""
        except Exception as exc:
            errors.append(f"{attempt} -> {type(exc).__name__}: {exc}")
    return False, " | ".join(errors) if errors else "runpp failed"


def get_bus_positions(net):
    if hasattr(net, "bus_geodata") and len(net.bus_geodata) > 0 and {"x", "y"}.issubset(net.bus_geodata.columns):
        geodata = net.bus_geodata.reindex(net.bus.index)
        if geodata[["x", "y"]].notna().all().all():
            return {int(bus): (float(geodata.at[bus, "x"]), float(geodata.at[bus, "y"])) for bus in net.bus.index}
    graph = nx.Graph()
    graph.add_nodes_from(int(bus) for bus in net.bus.index)
    for row in net.line.itertuples():
        graph.add_edge(int(row.from_bus), int(row.to_bus))
    for row in net.trafo.itertuples():
        graph.add_edge(int(row.hv_bus), int(row.lv_bus))
    return nx.spring_layout(graph, seed=0)


def plot_topology(net, agent_buses, title):
    positions = get_bus_positions(net)
    agent_buses = set(map(int, agent_buses))
    all_buses = set(map(int, net.bus.index))
    ext_buses = set(map(int, net.ext_grid["bus"].tolist()))
    load_buses = set(map(int, net.load["bus"].tolist()))
    sgen_buses = set(map(int, net.sgen["bus"].tolist()))
    other_buses = all_buses - agent_buses - load_buses - sgen_buses - ext_buses

    fig, ax = plt.subplots(figsize=(10, 6))
    for row in net.line.itertuples():
        x1, y1 = positions[int(row.from_bus)]
        x2, y2 = positions[int(row.to_bus)]
        ax.plot([x1, x2], [y1, y2], color="0.75", linewidth=1.0, zorder=1)
    for row in net.trafo.itertuples():
        x1, y1 = positions[int(row.hv_bus)]
        x2, y2 = positions[int(row.lv_bus)]
        ax.plot([x1, x2], [y1, y2], color="0.55", linewidth=1.8, zorder=1)

    categories = [
        ("Agent bus", agent_buses, "#d95f02", 120),
        ("Load bus", load_buses - agent_buses, "#1f78b4", 85),
        ("PV bus", sgen_buses - agent_buses, "#33a02c", 85),
        ("Slack/ext grid", ext_buses, "#e31a1c", 140),
        ("Other bus", other_buses, "#bdbdbd", 55),
    ]
    for label, buses, color, size in categories:
        buses = sorted(buses)
        if not buses:
            continue
        ax.scatter(
            [positions[bus][0] for bus in buses],
            [positions[bus][1] for bus in buses],
            s=size,
            c=color,
            label=label,
            edgecolors="black",
            linewidths=0.5,
            zorder=3,
        )
    for bus, (x, y) in positions.items():
        ax.text(x, y + 0.01, str(bus), fontsize=8, ha="center", va="bottom")
    ax.set_title(title)
    ax.set_xticks([])
    ax.set_yticks([])
    ax.legend(loc="upper left")
    ax.set_aspect("equal")
    plt.show()


def build_static_tables(net, agent_buses):
    slack_buses = set(net.ext_grid["bus"].astype(int).tolist())
    load_buses = set(net.load["bus"].astype(int).tolist())
    pv_buses = set(net.sgen["bus"].astype(int).tolist())
    agent_buses = set(map(int, agent_buses))
    bus_table = net.bus.reset_index(names="bus_id").copy()
    bus_table["is_slack"] = bus_table["bus_id"].isin(slack_buses)
    bus_table["has_load"] = bus_table["bus_id"].isin(load_buses)
    bus_table["has_pv"] = bus_table["bus_id"].isin(pv_buses)
    bus_table["is_agent_bus"] = bus_table["bus_id"].isin(agent_buses)
    bus_table["category"] = np.select(
        [
            bus_table["is_slack"],
            bus_table["is_agent_bus"],
            bus_table["has_load"] & bus_table["has_pv"],
            bus_table["has_load"],
            bus_table["has_pv"],
        ],
        ["slack", "agent", "prosumer", "load", "pv"],
        default="other",
    )
    line_table = net.line.reset_index(names="line_id").copy()
    trafo_table = net.trafo.reset_index(names="trafo_id").copy()
    static_summary = pd.Series(
        {
            "bus_count": int(len(bus_table)),
            "line_count": int(len(line_table)),
            "trafo_count": int(len(trafo_table)),
            "agent_bus_ids": sorted(agent_buses),
            "voltage_limit_min_pu": float(cfg.grid.v_min_pu),
            "voltage_limit_max_pu": float(cfg.grid.v_max_pu),
            "line_loading_limit_pct": float(cfg.grid.line_max_loading_pct),
        }
    )
    return static_summary, bus_table, line_table, trafo_table


In [ ]:
static_summary, bus_table, line_table, trafo_table = build_static_tables(raw_net, agent_bus_ids)
display(static_summary)
plot_topology(raw_net, agent_bus_ids, title=f"Topology: {cfg.grid.sb_code}")

display(bus_table[["bus_id", "name", "vn_kv", "type", "category", "has_load", "has_pv", "is_agent_bus", "is_slack"]])
display(line_table[["line_id", "from_bus", "to_bus", "length_km", "std_type", "max_i_ka", "max_loading_percent"]])
display(trafo_table[["trafo_id", "hv_bus", "lv_bus", "std_type", "sn_mva", "vn_hv_kv", "vn_lv_kv", "max_loading_percent"]])

project_extrema_df = pd.DataFrame(
    [
        {
            "series": "Agent total load (kW)",
            "max_value": float(project_year_df["agent_total_load_kw"].max()),
            "max_time": pd.Timestamp(project_year_df.loc[project_year_df["agent_total_load_kw"].idxmax(), "timestamp"]),
            "min_value": float(project_year_df["agent_total_load_kw"].min()),
            "min_time": pd.Timestamp(project_year_df.loc[project_year_df["agent_total_load_kw"].idxmin(), "timestamp"]),
        },
        {
            "series": "Agent total PV (kW)",
            "max_value": float(project_year_df["agent_total_pv_kw"].max()),
            "max_time": pd.Timestamp(project_year_df.loc[project_year_df["agent_total_pv_kw"].idxmax(), "timestamp"]),
            "min_value": float(project_year_df["agent_total_pv_kw"].min()),
            "min_time": pd.Timestamp(project_year_df.loc[project_year_df["agent_total_pv_kw"].idxmin(), "timestamp"]),
        },
        {
            "series": "Agent total net load (kW)",
            "max_value": float(project_year_df["agent_total_net_kw"].max()),
            "max_time": pd.Timestamp(project_year_df.loc[project_year_df["agent_total_net_kw"].idxmax(), "timestamp"]),
            "min_value": float(project_year_df["agent_total_net_kw"].min()),
            "min_time": pd.Timestamp(project_year_df.loc[project_year_df["agent_total_net_kw"].idxmin(), "timestamp"]),
        },
    ]
)
display(project_extrema_df)

daily = project_year_df.set_index("timestamp")[["agent_total_load_kw", "agent_total_pv_kw", "agent_total_net_kw"]].resample("D").mean()
ax = daily.plot(figsize=(14, 4.5), linewidth=1.0, title="Processed prosumer dataset daily mean power")
ax.set_ylabel("kW")
plt.tight_layout()
plt.show()

agent_metadata_table = pd.DataFrame(
    {
        "agent_idx": list(range(1, len(agent_bus_ids) + 1)),
        "agent_bus_id": agent_bus_ids,
        "agent_profile": list(metadata["agent_profiles"]),
        "pv_peak_kw": np.asarray(metadata["pv_peak_kw"], dtype=float),
        "ess_power_kw": np.asarray(metadata["ess_power_kw"], dtype=float),
        "ess_capacity_kwh": np.asarray(metadata["ess_capacity_kwh"], dtype=float),
    }
)
display(agent_metadata_table)


In [ ]:
def project_row_to_agent_vectors(row):
    load = np.array([float(row[f"load{i + 1}"]) for i in range(len(agent_bus_ids))], dtype=float)
    pv = np.array([float(row[f"pv{i + 1}"]) for i in range(len(agent_bus_ids))], dtype=float)
    net_load = load - pv
    return load, pv, net_load


def run_project_env_snapshot(row, batt_kw=None, label="project_env"):
    if batt_kw is None:
        batt_kw = np.zeros(len(agent_bus_ids), dtype=float)
    load_kw, pv_kw, base_net_load_kw = project_row_to_agent_vectors(row)
    net = build_simbench_net(cfg.grid.sb_code)
    bus_injection_map = {
        int(agent_bus_ids[i]): float(-(base_net_load_kw[i] + batt_kw[i]))
        for i in range(len(agent_bus_ids))
    }
    apply_bus_injections(net, bus_injection_map)
    ok, error = runpp_with_fallback(net, cfg.grid.pf_solver)
    if not ok:
        raise RuntimeError(f"Project snapshot failed at {row['timestamp']}: {error}")
    return {
        "label": label,
        "timestamp": pd.Timestamp(row["timestamp"]),
        "bus_df": net.res_bus.reset_index(names="bus_id").copy(),
        "line_df": net.res_line.reset_index(names="line_id").copy(),
        "trafo_df": net.res_trafo.reset_index(names="trafo_id").copy(),
        "agent_load_kw": load_kw,
        "agent_pv_kw": pv_kw,
        "agent_base_net_load_kw": base_net_load_kw,
        "agent_batt_kw": np.asarray(batt_kw, dtype=float),
        "agent_bus_injection_map_kw": bus_injection_map,
    }


project_peak_import_row = project_year_df.loc[project_year_df["agent_total_net_kw"].idxmax()]
project_peak_export_row = project_year_df.loc[project_year_df["agent_total_net_kw"].idxmin()]
battery_limit = float(cfg.env.max_charge_rate)
scenarios = [
    ("peak_import_no_batt", project_peak_import_row, np.zeros(len(agent_bus_ids), dtype=float)),
    ("peak_import_max_charge", project_peak_import_row, np.full(len(agent_bus_ids), battery_limit, dtype=float)),
    ("peak_import_max_discharge", project_peak_import_row, np.full(len(agent_bus_ids), -battery_limit, dtype=float)),
    ("peak_export_no_batt", project_peak_export_row, np.zeros(len(agent_bus_ids), dtype=float)),
    ("peak_export_max_charge", project_peak_export_row, np.full(len(agent_bus_ids), battery_limit, dtype=float)),
    ("peak_export_max_discharge", project_peak_export_row, np.full(len(agent_bus_ids), -battery_limit, dtype=float)),
]

summary_rows = []
for label, row, batt_kw in scenarios:
    state = run_project_env_snapshot(row, batt_kw=batt_kw, label=label)
    summary_rows.append(
        {
            "label": label,
            "timestamp": state["timestamp"],
            "vm_min": float(state["bus_df"]["vm_pu"].min()),
            "vm_max": float(state["bus_df"]["vm_pu"].max()),
            "max_line_loading_pct": float(state["line_df"]["loading_percent"].max()) if len(state["line_df"]) else np.nan,
            "max_trafo_loading_pct": float(state["trafo_df"]["loading_percent"].max()) if len(state["trafo_df"]) else np.nan,
            "agent_base_net_load_kw": np.round(state["agent_base_net_load_kw"], 4).tolist(),
            "agent_batt_kw": np.round(state["agent_batt_kw"], 4).tolist(),
            "agent_injection_map_kw": {int(k): round(float(v), 4) for k, v in state["agent_bus_injection_map_kw"].items()},
        }
    )

display(pd.DataFrame(summary_rows))
